In [64]:
using JuMP
using Gurobi
using Random
using Dualization
using Plots
using DataFrames
using CSV
import XLSX
import JSON

current_directory = @__DIR__
functions_directory = joinpath(current_directory, "functions")
data_dir = joinpath(current_directory, "data")
results_dir = joinpath(current_directory, "results")

# Include all the function files
# include(joinpath(functions_directory, "create_check_params.jl"))
# include(joinpath(functions_directory, "deterministic_equivalent.jl"))
# include(joinpath(functions_directory, "generate_cuts_from_dual.jl"))
include(joinpath(functions_directory, "load_model_starting_points.jl"))
# include(joinpath(functions_directory, "load_model_starting_points_OLD.jl"))
include(joinpath(functions_directory, "initialize_parameters.jl"))
# include(joinpath(functions_directory, "process_scenario_data_n_selected_with_MVP2.jl"))
include(joinpath(functions_directory, "process_scenario_data_n_selected_with_MVP2_demand_scaling.jl"))
include(joinpath(functions_directory, "process_scenario_data_n_selected.jl"))
# include(joinpath(functions_directory, "save_L_shaped_results.jl"))
include(joinpath(functions_directory, "select_random_scenarios.jl"))
include(joinpath(functions_directory, "create_vaccine_data.jl"))
# include(joinpath(functions_directory, "sub_problem.jl"))
# include(joinpath(functions_directory, "master_problem.jl"))
# include(joinpath(functions_directory, "create_vaccine_data_MMR_only.jl"))

create_vaccine_data

In [65]:
A, V, A_v, P, P_v, V_a, V_p, P_a, A_p, capacity_category, vaccine_category, antigen_category = create_vaccine_data()

(["Measles", "Mumps", "Rubella", "Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio", "HPV", "Rotavirus", "PCV"], ["M", "MR", "MMR", "TT", "HepB", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"], Dict("MMR" => ["Measles", "Mumps", "Rubella"], "Td" => ["Diphtheria", "Tetanus"], "PCV" => ["PCV"], "Rotavirus" => ["Rotavirus"], "DTwP-Hib" => ["Diphtheria", "Tetanus", "Pertussis", "Hib"], "IPV" => ["Polio"], "Hexa" => ["Diphtheria", "Tetanus", "Pertussis", "Hepatitis_B", "Hib", "Polio"], "TT" => ["Tetanus"], "HPV" => ["HPV"], "MR" => ["Measles", "Rubella"]…), ["AJ_Vaccines", "BB_NCIPD", "China_National", "Bharat_Biotech", "Bilthoven", "Biological_E", "GSK", "Haffkine_Bio", "LG_Chem", "Merck_Sharp", "Panacea_Biotec", "PT_Bio", "Sanofi", "Serum_Institute", "Pfizer"], Dict("MMR" => ["Serum_Institute", "GSK"], "Td" => ["Serum_Institute", "PT_Bio", "BB_NCIPD", "Biological_E"], "PCV" => ["Serum_Institute", "GSK", "Pfizer"], "Rotavirus" => 

In [66]:
starting_points_vect_F, starting_points_vect_I, starting_points_vect_S = load_model_starting_points(data_dir, 1, 1, A, V)

(Any[("Diphtheria", 1, 1), ("Tetanus", 1, 1), ("Pertussis", 1, 1), ("Hib", 1, 1), ("Hepatitis_B", 1, 1), ("Polio", 1, 1), ("Rotavirus", 1, 1), ("Measles", 1, 3), ("Mumps", 1, 3), ("Rubella", 1, 3), ("PCV", 1, 3), ("HPV", 1, 5)], Any[("Penta", 3.98056108e8), ("OPV", 2.796866e7), ("IPV", 4.10128001e8), ("PCV", 5.90379508e8), ("M", 2.80763726e8), ("MR", 8.79375105e8), ("MMR", 8.317184e7), ("TT", 7.6826e6), ("HepB", 3.278988e6), ("DT", 1.16834378e8), ("Td", 5.298998e6), ("DTwP", 3.0290394e7), ("DTwP-Hib", 4.3271992e7), ("Hexa", 8.6543984e7), ("HPV", 2.42285505e8), ("Rotavirus", 1.567269e8)], Any[("Diphtheria", 0.0), ("Hib", 0.0), ("Measles", 0.0), ("Mumps", 0.0), ("PCV", 0.0), ("Pertussis", 0.0), ("Polio", 0.0), ("Rotavirus", 0.0), ("Rubella", 0.0), ("Tetanus", 0.0), ("Hepatitis_B", 0.0), ("HPV", 0.0)])

In [67]:
T, T_initial, Δ, s_real, r, r_avg, r_producer_avg, g, h, l, f_profit, Γ, F_time_set, κ, L_lower_number, L_upper_number, delta, beta, zeta_vm, phi_vm_lower, phi_vm_upper, m_segments = initialize_parameters(data_dir, 1, 1, 10, 5, P, V, P_v, V_p, 1)

([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [1, 2, 3, 4, 5], Dict{Any, Any}("Sanofi" => 9.955e7, "Pfizer" => 6.842e7, "AJ_Vaccines" => 8.173e6, "Serum_Institute" => 8.767e8, "China_National" => 1.529e7, "BB_NCIPD" => 5.72e7, "Merck_Sharp" => 1.859e7, "Bilthoven" => 1.276e7, "Haffkine_Bio" => 1.122e8, "Biological_E" => 1.837e8…), Dict{Any, Any}(("PCV", "Serum_Institute", 6) => 2.29375, ("Rotavirus", "Bharat_Biotech", 2) => 0.9874999999999999, ("Rotavirus", "Serum_Institute", 2) => 0.9, ("Rotavirus", "Serum_Institute", 3) => 0.9, ("PCV", "GSK", 10) => 2.950390625, ("Rotavirus", "Bharat_Biotech", 10) => 0.9249999999999999, ("IPV", "Bilthoven", 5) => 1.9058333333333335, ("Rotavirus", "GSK", 10) => 2.1732915, ("DT", "BB_NCIPD", 3) => 0.175, ("OPV", "Serum_Institute", 9) => 0.13…), Dict{Any, Any}(("DT", 10) => 0.16999999999999998, ("IPV", 3) => 1.812625, ("DTwP", 2) => 0.177, ("HepB", 9) => 0.42874999999999996, ("Rotavirus", 7) => 1.3549595555555554, ("OPV", 3) => 0

In [68]:
f_profit[("HPV","Merck_Sharp",(1, 5))]

742.4969110180724

In [ ]:
# Scenarios_used, p_ω_test, p_ω_test_partial_2, Ω_test_partial_1, Ω_test_partial_2, partial_scenario, s_real_tilde, d_real_tilde, random_scenarios = process_scenario_data_n_selected_with_MVP2(current_directory, data_dir, 3, 3, A, T, P, 1, 10, 5, 1, 1, 1, 1,  true, 1, 19)
random_scenarios, s_real_tilde, d_real_tilde = process_scenario_data_n_selected_with_MVP2_demand_scaling(current_directory, data_dir, 1, 1, A, T, P, 1, 10, 5, 1, 1, 1, 1, true, 1, 0.02, 9)

In [ ]:
function check_constraint(Q, W, r_avg, zeta_vm, l_vp, f_profit, p, V_p, F_time_set, m)

    
    # Calculate the left-hand side (LHS) of the constraint
    LHS = 0.0
    for p in P
        for (t,tau) in F_time_set
            LHS += sum(r_avg[v, t] * (1 - zeta_vm[v, m]) * Q[v, p, (t, tau), m] for v in V_p[p])
        end
    end
    
    # Calculate the right-hand side (RHS) of the constraint
    RHS = 0.0
    for p in P
        for (t,tau) in F_time_set
            RHS += sum((1 + l[v, p]) * f_profit[v, p, t] * W[p, (t,tau)] for v in V_p[p])
        end
    end
    
    # Check if the constraint is satisfied
    is_satisfied = LHS >= RHS
    
    # Print the results for manual verification
    println("Final LHS: $LHS")
    println("Final RHS: $RHS")
    println("Is constraint satisfied? $is_satisfied")
    
    return LHS, RHS, is_satisfied
end


In [ ]:
Q = Dict()
for v in V
    for p in P_v[v]
        for (t,tau) in F_time_set
            for m in keys(m_segments)
                Q[v,p,(t,tau),m] = 3.68e7
            end
        end
    end
end

In [ ]:
W = Dict()
for p in P
    for (t,tau) in F_time_set
        W[p,(t,tau)] = 1
    end
end


In [ ]:
check_constraint(Q, W, r_avg, zeta_vm, l, f_profit, "Serum_Institute", V_p, F_time_set, 1)

In [ ]:
# Initialize an empty DataFrame to store the results
results_df = DataFrame(
    Producer = String[], 
    LHS = Float64[], 
    RHS = Float64[], 
    Satisfied = Bool[]
)

# Loop through each producer and run the check
for producer in P
    # Call the function and get the results
    LHS, RHS, is_satisfied = check_constraint(Q, W, r_avg, zeta_vm, l, f_profit, producer, V_p, F_time_set, 3)
    
    # Create a new row and push it to the DataFrame
    push!(results_df, (producer, LHS, RHS, is_satisfied))
end

# Print the final DataFrame
println(results_df)

using Pkg
Pkg.add("XLSX")
# Define the filename for your Excel file
filename = "constraint_results_segment_3.xlsx"

# Use XLSX.writetable to save the DataFrame to the Excel file
# The second argument is an array of tuples, where each tuple contains a sheet name and the data to write.
# Here we save the entire DataFrame to a sheet named "Results".
XLSX.writetable(filename, "Results" => results_df)

In [ ]:
# Define the directory where your data is located
current_directory = @__DIR__
data_dir = joinpath(current_directory, "data")

# Define the file path using a function similar to what you provided
prod_ratio_file_path = joinpath(data_dir, "cap_builder.xlsx")

sheet = "prop_by_manuf"

sheet_data = DataFrame(XLSX.readtable(prod_ratio_file_path, sheet))

# Select the first three columns from the DataFrame
# In Julia, DataFrame column indexing is 1-based, not 0-based
df_selected = sheet_data[:, [1,3,2]]

# Print the resulting DataFrame
df_selected


In [ ]:
    for p in P
        for v in V_p[p]

                selected_row = df_selected[(df_selected.Manufacturer .== p) .& (df_selected.Vaccine .== v), :]
                println(selected_row.Proportion[1])
                # proportion_value = selected_row.Proportion

        end
    end

# plotting and such

In [ ]:
import matplotlib.pyplot as plt

# # Adjust the length of both lists to match the shorter one
# min_length = min(len(lower_bounds), len(upper_bounds))

# # Trim both lists to the same length
# lower_bounds = lower_bounds[:min_length]
# upper_bounds = upper_bounds[:min_length]
# iterations = list(range(min_length))

# Plot the adjusted lower and upper bounds
plt.figure(figsize=(10, 6))
# Plot lower and upper bounds
plt.plot(new_updated_data["lb"], label="Lower Bound", marker="o", linestyle="-", color="blue")
plt.plot(new_updated_data["ub"], label="Upper Bound", marker="s", linestyle="--", color="red", alpha=0.7)

# Labels and title
plt.xlabel("Iteration")
plt.ylabel("Bound Value")
plt.title("L-Shaped with SEC/Knapsack/Feasbility Seeking - UG 3x3 scenarios")
plt.legend()
plt.grid(True)
# plt.savefig("Plot1.pdf")
# Show the plot
plt.show()


In [ ]:
new_updated_data = Dict(
    "lb" => [1.0924901117840355e10,1.4441619102439068e10,1.707874989968524e10,1.889379460554002e10,1.8897146707457954e10,2.1301584951597366e10,2.2490065439819847e10,2.5130791169482117e10,2.953345253546536e10,2.961493616408268e10,3.0592125248191513e10,3.1144739272642143e10,3.19782042060873e10,3.3907300787333546e10,3.3907300787333534e10,3.4060219382850582e10,3.4607469730074165e10,3.600211854863947e10,3.6002118548639465e10,3.826713843039722e10,3.826713843039719e10,3.826713843039719e10,3.826713843039722e10,3.8267138430397194e10,3.8267138430397194e10,3.850546395437657e10,3.988520812543393e10,3.9885208125433945e10,4.038079226879405e10,4.0728802909692276e10,4.109496857656064e10,4.158941694438292e10,4.2070917879357574e10,4.248699671739215e10,4.312365720956542e10,4.335804385923224e10,4.372181497551096e10,4.4109605868414276e10,4.494068267213822e10,4.4940682672138214e10,4.494068267213819e10,4.494068267213817e10,4.498129599277933e10,4.544670239940358e10,4.5446702399403496e10,4.544670239940358e10,4.569257286345092e10,4.582642967391234e10,4.633352587094907e10,4.6535459775862015e10,4.9575221624746e10,4.957522162474603e10,4.957522162474597e10,4.957522162474597e10,4.957522162474603e10,4.957522162474602e10,4.957522162474599e10,4.957522162474599e10,4.957522162474603e10,4.957522162474601e10,4.957522162474596e10,4.9575221624745995e10],
    "ub" => [5.072030934739337e11,3.412663482947864e11,3.412663482947864e11,3.171146089204061e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,2.0668013141015732e11,1.972331488078158e11,1.972331488078158e11,1.809594858895338e11,1.809594858895338e11,1.809594858895338e11,1.809594858895338e11,1.809594858895338e11,1.712990621890995e11,1.712990621890995e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.5959240285976184e11,1.501028246463881e11,1.4923320980154736e11,1.4336434263660605e11,1.4336434263660605e11,1.4336434263660605e11,1.3659797230508147e11,1.3659797230508147e11,1.3659797230508147e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11,1.3462084731755748e11]
)

In [ ]:
# Plotting
using Plots
# Accessing data from the dictionary using `new_updated_data["lb"]`
plot(new_updated_data["lb"],
     label="Lower Bound",
     marker=:circle,      # Matplotlib 'o' is typically :circle in Plots.jl
     linestyle=:solid,    # Matplotlib '-' is :solid
     color=:blue,
     linewidth=2,
     size=(1000, 600),    # Figure size in pixels (width, height)
     title="L-Shaped with SEC/Knapsack/Feasibility Seeking - UG 3x3 scenarios",
     xlabel="Iteration",
     ylabel="Bound Value",
     legend=:topright,    # Equivalent to matplotlib.pyplot.legend() default but explicitly set
     grid=true,
     framestyle=:box      # Adds a box around the plot area
)

# Use plot! to add the upper bounds to the *same* plot
plot!(new_updated_data["ub"],
      label="Upper Bound",
      marker=:rect,       # Matplotlib 's' is typically :rect (rectangle/square)
      linestyle=:dash,    # Matplotlib '--' is :dash
      color=:red,
      alpha=0.7,          # Transparency
      linewidth=2
)

# To save the plot uncomment the following line:
# savefig("plot2.pdf")


In [ ]:
import json

def extract_segments_with_z_value_1(json_data):
    """
    Extracts information where Z[vaccine, producer, time, segment] = 1.

    Args:
        json_data (str): A string containing the JSON data.

    Returns:
        dict: A dictionary containing the extracted information,
              structured by vaccine and producer.
    """
    data = json.loads(json_data)
    extracted_info = {}

    # Assuming 'Z' is a key in the main dictionary and its value is another dictionary
    # that holds vaccine, producer, time, and segment data.
    if 'Z' in data:
        z_data = data['Z']
        for vaccine, producers_data in z_data.items():
            for producer, time_data in producers_data.items():
                for time_str, segments_data in time_data.items():
                    time = int(time_str)
                    if 1 <= time <= 10:
                        for segment_str, value in segments_data.items():
                            segment = int(segment_str)
                            if 1 <= segment <= 3 and value == 1:
                                if vaccine not in extracted_info:
                                    extracted_info[vaccine] = {}
                                if producer not in extracted_info[vaccine]:
                                    extracted_info[vaccine][producer] = []
                                extracted_info[vaccine][producer].append({"Time": time, "Segment": segment})
    return extracted_info

# This is a placeholder for the content you fetched using file_content_fetcher.
# In your actual execution, you would replace this with the output from the tool.
# For demonstration purposes, I'm using a truncated version of the likely structure.

# Call the function with your actual JSON content
result = extract_segments_with_z_value_1(json_content_from_file)

# Print the results in a readable format
for vaccine, producers_data in result.items():
    print(f"For {vaccine} vaccine:")
    for producer, segments_list in producers_data.items():
        print(f"* **{producer}:**")
        for entry in segments_list:
            print(f"    * Time: {entry['Time']}, Segment: {entry['Segment']}")

In [ ]:
for trial in range(1,12):
    # Load the JSON file
    filename = f"results/DE_demand_increase_sensitivity/Balance Model/10 trial inc demand 3 segments/MVP_DE_results_T_10_delta_5_scen_1_trial_{trial}_inv_1_cap._1_cap.inc._1.json"
    with open(filename) as json_file:
        data = json.load(json_file)